# Agentic Extraction: Schema-Constraint Validation & Model Escalation

Agentic extraction always validates the agent's output against the **Pydantic
model** generated from the class JSON Schema, so `enum`, `pattern`, numeric
bounds and `minItems`/`maxItems` violations are fed back to the agent for
self-correction during extraction.

The optional `extraction.agentic.validation` block adds two capabilities on top
of that:

1. **Full JSON-Schema validation of the final result** — most importantly the
   `format` keyword (`date`, `date-time`, `email`, `uri`, `uuid`, ...), which
   the generated Pydantic model does **not** enforce. All violations are
   collected in one pass with readable field paths so the agent (or an
   escalation pass) can fix them together instead of one-at-a-time.
2. **Bounded model escalation** — when validation still fails, optionally
   re-extract **once** with a stronger (more expensive) model, seeded with the
   best-effort result and the list of violations. Still far cheaper than human
   review.

This notebook demonstrates the behavior **deterministically** — `validate_extraction`
is pure (no AWS), and for the escalation scenario we *mock* the Bedrock call so
we can force a low-quality first attempt and a corrected second attempt. In
production the same loop runs against the real model — no code changes, just the
config flags shown below.

> **Related config keys** (under `extraction.agentic.validation:`):
> - `enabled` (default `false`)
> - `check_formats` (default `true`)
> - `fail_action` — `warn` | `escalate` | `reject` (default `escalate`)
> - `escalation_model` (per-class override: `x-aws-idp-extraction-escalation-model`)


## 1. Imports

In [1]:
import json
import logging
from unittest.mock import patch

from idp_common.extraction.service import ExtractionService
from idp_common.extraction.validation import validate_extraction

# Surface the validation / escalation log messages so we can watch them fire.
logging.basicConfig(level=logging.WARNING)
logging.getLogger("idp_common.extraction").setLevel(logging.INFO)

print("Imports OK")

Imports OK


## 2. Define a class schema with rich constraints

A realistic invoice class: a `pattern` on the id, a `format: date` on the issue
date, an `enum` status, and a `minItems` line-item array. The
`x-aws-idp-*` extensions are IDP-specific and are stripped before validation.

In [2]:
SCHEMA = {
    "type": "object",
    "$id": "Invoice",
    "x-aws-idp-document-type": "Invoice",
    "required": ["invoice_id", "issue_date"],
    "properties": {
        "invoice_id": {"type": "string", "pattern": "^INV-[0-9]+$"},
        "issue_date": {"type": "string", "format": "date"},   # ISO-8601 YYYY-MM-DD
        "customer_email": {"type": "string", "format": "email"},
        "status": {
            "type": "string",
            "enum": ["paid", "due", "void"],
            "x-aws-idp-evaluation-method": "EXACT",
        },
        "lines": {
            "type": "array",
            "minItems": 1,
            "items": {
                "type": "object",
                "properties": {"description": {"type": "string"}, "amount": {"type": "number"}},
            },
        },
    },
}

def valid_doc():
    return {
        "invoice_id": "INV-1024",
        "issue_date": "2024-01-02",
        "customer_email": "ap@acme.example",
        "status": "paid",
        "lines": [{"description": "Widgets", "amount": 100.0}],
    }

print("Schema + helper defined")

Schema + helper defined


## 3. A fully valid extraction passes

In [3]:
report = validate_extraction(valid_doc(), SCHEMA)
print("valid       :", report.valid)
print("error_count :", len(report.errors))
print("failed      :", sorted(report.failed_top_level_fields))
print()
print(report.agent_feedback())

valid       : True
error_count : 0
failed      : []

All extracted fields satisfy the schema constraints.


## 4. An invalid extraction — all violations reported at once

We corrupt four fields: a bad `pattern`, a `MM/DD/YYYY` date (not ISO-8601, so a
`format` failure the Pydantic model would miss), an out-of-`enum` status, and an
empty `minItems` array. Note they are **all** surfaced in a single pass, each
with a field path — exactly the message the agent receives to self-correct.

In [4]:
bad = {
    "invoice_id": "1024",            # pattern fail (missing INV- prefix)
    "issue_date": "01/02/2024",      # format fail (not ISO-8601)
    "customer_email": "not-an-email",# format fail
    "status": "pending",             # enum fail
    "lines": [],                     # minItems fail
}
report = validate_extraction(bad, SCHEMA)
print("valid       :", report.valid)
print("error_count :", len(report.errors))
print("failed      :", sorted(report.failed_top_level_fields))
print()
print(report.agent_feedback())

valid       : False
error_count : 5
failed      : ['customer_email', 'invoice_id', 'issue_date', 'lines', 'status']

The extraction violates the following schema constraints. Fix each one using the available tools and keep all other data:
  - customer_email: 'not-an-email' is not a 'email'
  - invoice_id: '1024' does not match '^INV-[0-9]+$'
  - issue_date: '01/02/2024' is not a 'date'
  - lines: [] should be non-empty
  - status: 'pending' is not one of ['paid', 'due', 'void']


## 5. The `check_formats` switch

JSON-Schema `format: date` means ISO-8601 (`YYYY-MM-DD`). The **default
extraction prompt asks the model for `MM/DD/YYYY`**, which is *not* a valid
`date` — so it fails format validation. If your schema uses `format: date` for
non-ISO dates, either set `check_formats: false` or use a `pattern` instead of
`format`.

In [5]:
doc = valid_doc()
doc["issue_date"] = "01/02/2024"  # MM/DD/YYYY

with_formats = validate_extraction(doc, SCHEMA, check_formats=True)
without_formats = validate_extraction(doc, SCHEMA, check_formats=False)

print("check_formats=True  -> valid:", with_formats.valid)
print("check_formats=False -> valid:", without_formats.valid)

check_formats=True  -> valid: False
check_formats=False -> valid: True


## 6. The in-loop validator the agent uses

`ExtractionService._build_schema_validator()` returns the callback that runs
*inside* the agent loop: it takes the extracted dict and returns
`(is_valid, feedback)`. When validation is disabled it returns `None` (zero
overhead, no behavior change).

In [6]:
def make_service(validation=None, escalation_class_override=None):
    agentic = {"enabled": True}
    if validation is not None:
        agentic["validation"] = validation
    schema = dict(SCHEMA)
    if escalation_class_override:
        schema = {**SCHEMA, "x-aws-idp-extraction-escalation-model": escalation_class_override}
    config = {
        "extraction": {"model": "us.amazon.nova-pro-v1:0", "agentic": agentic},
        "classes": [schema],
    }
    svc = ExtractionService(region="us-west-2", config=config)
    svc._class_schema = schema  # normally set during process_document_section
    return svc

svc_off = make_service(validation={"enabled": False})
print("validation disabled -> callback is:", svc_off._build_schema_validator())

svc_on = make_service(validation={"enabled": True})
validator = svc_on._build_schema_validator()
ok, _ = validator(valid_doc())
bad_ok, feedback = validator({**valid_doc(), "status": "pending"})
print("valid doc   -> is_valid:", ok)
print("invalid doc -> is_valid:", bad_ok)
print("feedback the agent receives:\n", feedback)

INFO:idp_common.extraction.service:Initialized extraction service with model us.amazon.nova-pro-v1:0


INFO:idp_common.extraction.service:Initialized extraction service with model us.amazon.nova-pro-v1:0


validation disabled -> callback is: None
valid doc   -> is_valid: True
invalid doc -> is_valid: False
feedback the agent receives:
 The extraction violates the following schema constraints. Fix each one using the available tools and keep all other data:
  - status: 'pending' is not one of ['paid', 'due', 'void']


## 7. `fail_action` — what happens when validation still fails

`_validate_and_maybe_escalate` is the service-level gate applied to the final
(post-merge) result. With **`warn`** it records a `validation` metadata block and
proceeds; with **`reject`** it flips `parsing_succeeded` to `False` so
downstream / HITL can act. Neither calls a model, so both run with no AWS.

In [7]:
class FakeSection:
    class_label = "Invoice"

bad = {**valid_doc(), "status": "pending"}  # single enum violation

for action in ("warn", "reject"):
    svc = make_service(validation={"enabled": True, "fail_action": action})
    fields, data, meta, metering, parsing_ok = svc._validate_and_maybe_escalate(
        extracted_fields=bad,
        structured_data=None,
        data_model=None,
        model_id="us.amazon.nova-pro-v1:0",
        message_prompt="(prompt)",
        agentic_images=[],
        custom_instruction=None,
        section_info=FakeSection(),
        parsing_succeeded=True,
    )
    print(f"fail_action={action!r:9} -> parsing_succeeded={parsing_ok}  "
          f"escalated={meta['escalated']}  valid={meta['valid']}  "
          f"failed_fields={meta['failed_fields']}")

INFO:idp_common.extraction.service:Initialized extraction service with model us.amazon.nova-pro-v1:0


INFO:idp_common.extraction.service:Initialized extraction service with model us.amazon.nova-pro-v1:0


fail_action='warn'    -> parsing_succeeded=True  escalated=False  valid=False  failed_fields=['status']
fail_action='reject'  -> parsing_succeeded=False  escalated=False  valid=False  failed_fields=['status']


## 8. `fail_action: escalate` — re-extract only the failing fields

When the first pass fails, escalation re-extracts **only the failing top-level
fields** with a stronger model (precedence: per-class
`x-aws-idp-extraction-escalation-model` → global `escalation_model` → the
extraction model itself) and merges the corrected fields back. Scoping to the
failing fields keeps the schema/prompt/output small — cheaper and faster than
re-running the whole section — and the already-valid fields are preserved.

We mock both the subset-model builder and `structured_output` so the "stronger
model" deterministically returns a *corrected* value for just the failing field,
with no Bedrock call. Note the audit metadata: `escalation_scope`,
`escalation_fields`, and `resolved_by_escalation`.

In [8]:
from unittest.mock import patch

from pydantic import BaseModel


# A tiny stand-in for the dynamic Pydantic model the service normally builds.
class InvoiceModel(BaseModel):
    invoice_id: str
    issue_date: str
    customer_email: str | None = None
    status: str | None = None
    lines: list | None = None


# The subset model the service would generate for just the failing field.
class StatusOnlyModel(BaseModel):
    status: str | None = None


svc = make_service(
    validation={
        "enabled": True,
        "fail_action": "escalate",
        "escalation_model": "us.anthropic.claude-opus-4-8",
    },
)

# First pass is good except for an invalid `status` enum value.
first_pass_bad = {
    "invoice_id": "INV-1024",
    "issue_date": "2024-01-02",
    "customer_email": "ap@acme.example",
    "status": "pending",                       # <- only this field is invalid
    "lines": [{"description": "Widgets", "amount": 100.0}],
}

captured = {}


def fake_structured_output(*args, **kwargs):
    sub_model = kwargs["data_format"]
    captured["model_id"] = kwargs["model_id"]
    captured["scoped_fields"] = sorted(sub_model.model_fields.keys())
    # The stronger model corrects ONLY the field it was given.
    return sub_model(status="void"), {
        "metering": {"ExtractionEscalation/bedrock/strong": {"inputTokens": 80}}
    }


# Patch the subset-model builder + the extraction call (no Bedrock needed).
with (
    patch(
        "idp_common.extraction.service.create_pydantic_model_from_json_schema",
        return_value=StatusOnlyModel,
    ),
    patch("idp_common.extraction.service.structured_output", fake_structured_output),
):
    fields, data, meta, metering, parsing_ok = svc._validate_and_maybe_escalate(
        extracted_fields=first_pass_bad,
        structured_data=InvoiceModel(**first_pass_bad),
        data_model=InvoiceModel,
        model_id="us.amazon.nova-pro-v1:0",
        message_prompt="(prompt)",
        agentic_images=[],
        custom_instruction="Extract the invoice.",
        section_info=FakeSection(),
        parsing_succeeded=True,
    )

print("escalation model used :", captured["model_id"])
print("re-extracted ONLY     :", captured["scoped_fields"], "(field-subset scope)")
print()
print("status corrected      :", fields["status"], "(was 'pending')")
print("invoice_id preserved  :", fields["invoice_id"])
print("lines preserved       :", fields["lines"])
print()
print("=== audit metadata (metadata.validation) ===")
print(json.dumps(meta, indent=2))

INFO:idp_common.extraction.service:Initialized extraction service with model us.amazon.nova-pro-v1:0


INFO:idp_common.extraction.service:Escalating extraction for 'Invoice' to model us.anthropic.claude-opus-4-8 (scope=field-subset, fields=['status'])


escalation model used : us.anthropic.claude-opus-4-8
re-extracted ONLY     : ['status'] (field-subset scope)

status corrected      : void (was 'pending')
invoice_id preserved  : INV-1024
lines preserved       : [{'description': 'Widgets', 'amount': 100.0}]

=== audit metadata (metadata.validation) ===
{
  "valid": true,
  "error_count": 0,
  "failed_fields": [],
  "errors": [],
  "check_formats": true,
  "fail_action": "escalate",
  "escalated": true,
  "initial_error_count": 1,
  "initial_failed_fields": [
    "status"
  ],
  "escalation_model": "us.anthropic.claude-opus-4-8",
  "escalation_scope": "field-subset",
  "escalation_fields": [
    "status"
  ],
  "resolved_by_escalation": true
}


## 9. Escalation-model precedence (per-class override beats global)

In [9]:
svc = make_service(
    validation={"enabled": True, "escalation_model": "global-model"},
    escalation_class_override="class-model",
)
print("per-class override present -> resolves to:", svc._resolve_escalation_model())

svc = make_service(validation={"enabled": True, "escalation_model": "global-model"})
print("only global set            -> resolves to:", svc._resolve_escalation_model())

svc = make_service(validation={"enabled": True})
print("neither set                -> resolves to:", svc._resolve_escalation_model(),
      "(escalation becomes a plain retry with the extraction model)")

INFO:idp_common.extraction.service:Initialized extraction service with model us.amazon.nova-pro-v1:0


INFO:idp_common.extraction.service:Initialized extraction service with model us.amazon.nova-pro-v1:0


INFO:idp_common.extraction.service:Initialized extraction service with model us.amazon.nova-pro-v1:0


per-class override present -> resolves to: class-model
only global set            -> resolves to: global-model
neither set                -> resolves to: None (escalation becomes a plain retry with the extraction model)


## 10. Summary

| Scenario | `validation.enabled` | `fail_action` | Outcome |
|----------|----------------------|---------------|---------|
| 3 | `true` | — | valid result passes, no action |
| 4 | `true` | — | all violations reported in one pass with field paths |
| 7 (warn) | `true` | `warn` | `metadata.validation` recorded, processing continues |
| 7 (reject) | `true` | `reject` | `parsing_succeeded=false` (route to HITL) |
| 8 (escalate) | `true` | `escalate` | re-extract **only the failing fields** with a stronger model; merge back, keep if valid/fewer-errors |

**To enable in your deployment**, set this under `extraction.agentic:` in your
config (off by default — no behavior change on upgrade):

```yaml
extraction:
  agentic:
    enabled: true
    validation:
      enabled: true
      check_formats: true          # ISO-8601 dates; see the caveat in section 5
      fail_action: escalate        # warn | escalate | reject
      escalation_model: "us.anthropic.claude-opus-4-8"
```

Every section's `metadata` records the `extraction_model` used and a
`metadata.validation` block (rules enforced, before/after error counts,
`escalation_scope`/`escalation_fields`/`resolved_by_escalation`) for auditing.

Per class, override the escalation model with the
`x-aws-idp-extraction-escalation-model` schema extension. The validation outcome
is written to `metadata.validation` in each section's extraction result
(`valid`, `error_count`, `failed_fields`, `errors`, `escalated`,
`escalation_model`).

See `lib/idp_common_pkg/idp_common/extraction/README.md` →
*"Schema-Constraint Validation and Model Escalation"* for full details.
